In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE = "workspace.complaints.bronze_complaints"
SILVER = "workspace.complaints.silver_complaints"
QUARANTINE = "workspace.complaints.silver_quarantine"

df_bronze = spark.table(BRONZE)

# Safely parse timestamps without throwing CANNOT_PARSE_TIMESTAMP
df_parsed = (
    df_bronze
    .withColumn("parsed_date_received", F.expr("try_to_timestamp(date_received)"))
    .withColumn("parsed_date_sent", F.expr("try_to_timestamp(date_sent_to_company)"))
)

# Quarantine check: complaint_id must exist and date_received must parse
is_quarantine = (
    F.col("complaint_id").isNull()
    | (F.trim(F.col("complaint_id").cast("string")) == "")
    | F.col("parsed_date_received").isNull()
)

# Route quarantined records
df_bad = (
    df_parsed.filter(is_quarantine)
    .withColumn("quarantine_reason", F.lit("Invalid complaint_id or unparseable date_received"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
)

if df_bad.count() > 0:
    (
        df_bad.drop("parsed_date_received", "parsed_date_sent")
        .write.format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE)
    )

# Valid records: replace original date columns with parsed timestamps
df_valid = (
    df_parsed.filter(~is_quarantine)
    .withColumn("date_received", F.col("parsed_date_received"))
    .withColumn("date_sent_to_company", F.col("parsed_date_sent"))
    .drop("parsed_date_received", "parsed_date_sent")
)

# Deduplicate batch on complaint_id keeping latest load_timestamp
w = Window.partitionBy("complaint_id").orderBy(F.col("load_timestamp").desc())
df_clean = (
    df_valid
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

# Idempotent MERGE into Silver
if not spark.catalog.tableExists(SILVER):
    df_clean.write.format("delta").saveAsTable(SILVER)
    print(f"Created Silver table with {df_clean.count()} rows.")
else:
    silver_table = DeltaTable.forName(spark, SILVER)
    (
        silver_table.alias("t")
        .merge(
            df_clean.alias("s"),
            "t.complaint_id = s.complaint_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Successfully merged {df_clean.count()} records into Silver.")

dbutils.notebook.exit("SUCCESS")

In [0]:
# 1. Check Silver records count
print("Silver row count:", spark.table("workspace.complaints.silver_complaints").count())

# 2. Check latest execution audit log
display(spark.table("workspace.complaints.pipeline_execution_logs").orderBy(F.col("start_time").desc()).limit(5))

# 3. Check updated high-water mark
display(spark.table("workspace.complaints.pipeline_watermark"))

In [0]:
from pyspark.sql import functions as F

bronze_count = spark.table("workspace.complaints.bronze_complaints").count()
silver_count = spark.table("workspace.complaints.silver_complaints").count()
quarantine_count = spark.table("workspace.complaints.silver_quarantine").count()

print("=" * 50)
print(f"Bronze Total Records:     {bronze_count}")
print(f"Silver Total Records:     {silver_count}")
print(f"Quarantine Total Records: {quarantine_count}")
print("=" * 50)

# Check schema and sample data from Silver
display(spark.table("workspace.complaints.silver_complaints").limit(5))

In [0]:
display(spark.sql("SHOW TABLES IN workspace.complaints"))